In [9]:
from pathlib import Path

MODEL_DIR  = Path("model/BD_llama_6heads_1epoch_4layers")
DATA_DIR   = Path("data/BD_llama_inital")
REMAP_PATH = DATA_DIR / "old_to_new.json"
TOKENS_PATH = DATA_DIR / "bios_postreduce.bin"

SAE_PATH = "/Users/efmac/Code/Project Code/CRL-Interp/Interp_LM4/sae_runs/sweep-n66crzzw/mult16_l05_lr3e-05_ep50_n10000/final"

### Load Model, Data, and Tokenizer

In [10]:
from condensed_tokenizer import CondensedTokenizer
from bio_sampler import BioSampler

tokenizer = CondensedTokenizer.from_remap_path(REMAP_PATH)
sampler   = BioSampler(DATA_DIR / "people.json", fields=("birthday",), seed=0)

print(f"vocab_size = {tokenizer.vocab_size}, eos_token_id = {tokenizer.eos_token_id}")
print(f"{len(sampler.people):,} people, {sampler.n_templates} templates/person\n")

# Specific person + specific template
text = sampler.render(sampler.people[0], exposure_idx=0)
print("render(people[0], 0) →", repr(text))
print("encode →", tokenizer.encode(text)[:15], "...\n")

# Random bio
draw = sampler.sample()
print(f"sample() → person id={draw['person']['id']}, template={draw['exposure_idx']}")
print("text →", repr(draw["text"]))

vocab_size = 1836, eos_token_id = 1835
50,000 people, 46 templates/person

render(people[0], 0) → ' Gabriella Ella Rigby was born on February 18, 1816.'
encode → [870, 83, 882, 663, 5, 1273, 267, 80, 536, 52, 487, 237, 1, 237, 256] ...

sample() → person id=50494, template=26
text → ' Marco Jackson Rowland arrived in this world on December 24, 1717, a day to be remembered.'


In [11]:
import torch
from transformers import LlamaForCausalLM
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformer_lens.loading_from_pretrained import convert_llama_weights


def pick_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


device = pick_device()
dtype = torch.float32

hf_model = LlamaForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=dtype)
hf_model.eval()
assert hf_model.config.vocab_size == tokenizer.vocab_size, (
    f"checkpoint vocab {hf_model.config.vocab_size} != remap vocab "
    f"{tokenizer.vocab_size} — wrong old_to_new.json for this model."
)

# Build the TL config from the HF config so dims match our custom
# 4-layer / hidden=384 / vocab=1836 model (from_pretrained would have used
# the Llama-2-7b template config and tried to read layer 4 of a 4-layer model).
hf_cfg = hf_model.config
tl_cfg = HookedTransformerConfig(
    n_layers = hf_cfg.num_hidden_layers,
    d_model = hf_cfg.hidden_size,
    d_head = hf_cfg.hidden_size // hf_cfg.num_attention_heads,
    n_heads = hf_cfg.num_attention_heads,
    d_mlp= hf_cfg.intermediate_size,
    d_vocab=hf_cfg.vocab_size,
    n_ctx=hf_cfg.max_position_embeddings,
    act_fn="silu",
    normalization_type="RMS",
    gated_mlp=True,
    positional_embedding_type="rotary",
    rotary_base=int(getattr(hf_cfg, "rope_theta", 10000.0)),
    rotary_dim=hf_cfg.hidden_size // hf_cfg.num_attention_heads,
    final_rms=True,
    tie_word_embeddings=hf_cfg.tie_word_embeddings,
    initializer_range=hf_cfg.initializer_range,
    n_key_value_heads=hf_cfg.num_key_value_heads,
    device=device,
)

# Pre-tokenize everything you feed the model; don't attach the tokenizer.
# TL only calls into the tokenizer for `model(str)` / `to_tokens` / `to_string`,
# none of which we use — we always pass token ids directly.
state_dict = convert_llama_weights(hf_model, tl_cfg)
model = HookedTransformer(tl_cfg)
model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()
print(f"Loaded on {device}: n_layers={model.cfg.n_layers}, d_model={model.cfg.d_model}, "
      f"n_heads={model.cfg.n_heads}, d_vocab={model.cfg.d_vocab}")

Loading weights: 100%|██████████| 38/38 [00:00<00:00, 56140.74it/s]


Moving model to device:  mps
Loaded on mps: n_layers=4, d_model=384, n_heads=6, d_vocab=1836


In [12]:
def show_tokens(ids, tokenizer, addOne=False):
    """Print each token with its position and decoded text (with repr so
    leading spaces / newlines stay visible).

    addOne=True shifts the displayed index by 1, so positions match what
    the model sees after a BOS/EOS is prepended to `ids` downstream.
    """
    if hasattr(ids, "tolist"):
        ids = ids.tolist()
    if ids and isinstance(ids[0], list):
        ids = ids[0]   # unwrap [1, N] batch

    offset = 1 if addOne else 0
    id_w = max(len(str(t)) for t in ids)
    idx_w = len(str(len(ids) - 1 + offset))
    print(f"{'idx':>{idx_w}} | {'id':>{id_w}} | text")
    print("-" * (idx_w + id_w + 12))
    for i, t in enumerate(ids):
        print(f"{i + offset:>{idx_w}} | {t:>{id_w}} | {tokenizer.decode([t])!r}")


### Exploring SAE 

#### Setup: load the SAE

The SAE checkpoint is loaded once and reused by every subsequent cell.

In [ ]:
from evalSAE import load_sae

sae = load_sae(SAE_PATH, device)
HOOK = "blocks.1.hook_mlp_out"
print(f"d_sae = {sae.cfg.d_sae}, hook = {HOOK}")

#### Option A — View precomputed dashboard&nbsp;&nbsp; ← RECOMMENDED

Run `computeInference.py` on the HPC (`sbatch submit_job_psc.sh computeInference.py`).
That writes `inference/<sae_name>/dashboard.html`. Then on this laptop:

```bash
scp -r friedmae@bridges2:Interp_LM4/inference ./
```

Run the cell below to display the dashboard inline.

In [ ]:
from pathlib import Path
from IPython.display import IFrame, display

# The output dir is named after the SAE checkpoint's parent directory.
sae_name = Path(SAE_PATH).parent.name
DASHBOARD_HTML = Path("inference") / sae_name / "dashboard.html"
print("dashboard:", DASHBOARD_HTML)
print("exists:   ", DASHBOARD_HTML.exists())

display(IFrame(str(DASHBOARD_HTML), width=1100, height=800))

#### Option B — Compute the dashboard locally

Use this only on a GPU machine or when you can wait 10–30+ minutes on MPS.
It runs the model + SAE forward over the full bio corpus and writes the
same `dashboard.html` that Option A reads. For fast iteration, pass
`features=range(100)` to `make_dashboard` to render a feature subset.

In [ ]:
from pathlib import Path
from sae_explorer import build_index_corpus

DASHBOARD_DIR = Path("inference") / Path(SAE_PATH).parent.name
tokens = build_index_corpus(
    sampler,
    tokenizer,
    n_per_person=2,
    context_size=64,
    seed=0,
    cache_path=DASHBOARD_DIR / "index_corpus.pt",
)
print("corpus shape:", tokens.shape)  # expect [N, 64]

In [ ]:
from sae_explorer import make_dashboard

# Full pass: every feature in the SAE over the index corpus.
# Expect 10–30 min on MPS, much faster on CUDA.
# Pass features=range(100) (or similar) to render a subset while iterating.
out_html = make_dashboard(
    model,
    sae,
    tokens.to(device),
    tokenizer,
    out_dir=DASHBOARD_DIR,
    hook_name=HOOK,
)
print("open in browser:", out_html)

#### Direct Logit Attribution (DLA)

Project a feature's decoder direction through the model's unembed matrix
to see which output tokens it linearly promotes (top) and suppresses
(bottom). The dashboard shows this per-feature; this cell lets you
inspect features programmatically.

Note: DLA ignores downstream attention/MLP and any layernorm scaling.
It's a cheap intuition, not a full causal account — pair with `steer()`
below if you want the real effect on predictions.

In [ ]:
from sae_explorer import dla

# Replace feature_idx with one you found interesting in the dashboard.
result = dla(sae, model, tokenizer, feature_idx=0, k=10)
print("=== TOP (promoted) ===")
for r in result["top"]:
    print(f"  {r['logit_delta']:+.3f}  {r['text']!r:>20}  (id={r['token_id']})")
print("\n=== BOTTOM (suppressed) ===")
for r in result["bottom"]:
    print(f"  {r['logit_delta']:+.3f}  {r['text']!r:>20}  (id={r['token_id']})")

#### Causal probes — `steer()`

Boost one feature's decoder direction at the hook, compare next-token
predictions with and without the boost. Fast even on a laptop (one
forward pass on a short input).

In [ ]:
from sae_explorer import steer

# Pick a feature from the dashboard's dropdown and probe it causally.
result = steer(
    model, sae, tokenizer,
    text=" Gabriella Ella Rigby was born on",
    feature_idx=0,         # replace with a feature that looked interesting
    scale=5.0,
    hook_name=HOOK,
)
for k, rows in result.items():
    print(f"\n=== {k} ===")
    for r in rows:
        print(f"  {r['logit']:+.2f}  {r['text']!r}  (id={r['token_id']})")

### Exploring SAE - Specific Example

In [ ]:
sample = sampler.sample()
sample

In [ ]:
ids = [tokenizer.eos_token_id] + tokenizer.encode(sample["text"]) #Add a leading EOS so the first content token has a fresh attention context,
input_tokens = torch.tensor([ids], device=device)
show_tokens(tokenizer.encode(sample["text"]), tokenizer, addOne=True)